# 02 — Feature Matrix Construction

**Goal:** Convert 900 × 8-tool raw outputs into a single tidy feature matrix (`data/processed/feature_matrix.parquet`).

**Construction order:**
1. DefenseFinder (defence + anti-defence systems)
2. PADLOC (defence systems)
3. Merge DF + PADLOC via `system_name_map.csv`
4. ResFinder (ARG counts)
5. ICEberg (IME/ICE counts, with BLAST coverage filter)
6. BacMet (HMRG counts, with BLAST coverage filter)
7. ISEScan (IS element counts by family)
8. MLST + metadata (join keys and labels)
9. Derived features: ratios, sparsity filter, ARG burden tertile labels
10. Save to parquet

Each section is explained before the code. Run cells in order — later sections depend on earlier ones.

## Imports and paths

All paths are relative to the notebook's location (`notebooks/`). The `..` prefix navigates up to the project root.

`numpy` is imported but not used until ratio computation — importing at the top avoids hunting for it later.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Project layout ──────────────────────────────────────────────────────────
ROOT    = Path("..")                   # project root (one level up from notebooks/)
INTERIM = ROOT / "data" / "interim"   # per-species tool outputs
PROC    = ROOT / "data" / "processed" # final feature matrix lands here
CONFIG  = ROOT / "config"

PROC.mkdir(parents=True, exist_ok=True)  # create processed/ if missing

# ── Six ESKAPE species, matching interim/ subdirectory names ─────────────────
SPECIES = [
    "abaumannii",
    "ecloaceae",
    "efaecium",
    "kpneumoniae",
    "paeruginosa",
    "saureus",
]

# ── Accession normalisation ───────────────────────────────────────────────
# Tool output dirs/files use GCF_XXXXXXXXX_V (underscore before version).
# NCBI canonical form is GCF_XXXXXXXXX.V (dot before version).
# All downstream joins use the canonical dot form.
def norm_acc(s: str) -> str:
    """GCF_000505685_1 → GCF_000505685.1"""
    prefix, version = s.rsplit("_", 1)   # split on LAST underscore only
    return f"{prefix}.{version}"

print("Paths OK")
print(f"  interim : {INTERIM.resolve()}")
print(f"  processed: {PROC.resolve()}")

## Section 1 — Parse DefenseFinder outputs

### Why this step

DefenseFinder is the primary source for defence system calls. Each genome has its own subdirectory under `interim/{species}/defensefinder/{accession}/`. Inside is `defense_finder_systems.tsv` with one row per detected system **instance** (a genome with three RM systems has three rows).

Key columns we use:
- `subtype` — the specific system name (e.g. `RM_Type_I`, `BREX_I`, `SspBCDE`)
- `activity` — `"Defense"` or `"Antidefense"` (anti-defence is output when `--antidefensefinder` is passed to DefenseFinder)

We parse defence and anti-defence into separate long-form tables. A long-form table has one row per (genome, system) observation — potentially multiple rows for the same genome if it carries multiple instances. We deduplicate later when building presence/absence columns; we keep the raw counts for the count columns.

### Mapping DF subtype names → canonical names

DefenseFinder and PADLOC use different names for the same systems (e.g. DF calls it `SspBCDE`, PADLOC calls it `PT_SspABCD`). The `system_name_map.csv` resolves this. We load it first and build a lookup dictionary: `df_subtype → canonical_name`.

Systems where `source == 'exclude'` are dropped (only `DMS_other`, a catch-all present in 886/900 genomes — uninformative).

In [ ]:
# ── Load system name map ─────────────────────────────────────────────────────
name_map = pd.read_csv(CONFIG / "system_name_map.csv")

print(f"system_name_map: {len(name_map)} rows")
print(name_map["source"].value_counts().to_string())
print()

# Build lookup: df_subtype → canonical_name
# Exclude 'exclude' rows (DMS_other catch-all — 886/900 genomes, uninformative).
# Keep 'both', 'df_only', 'df_antidefense' — all have a valid df_subtype entry.
# Note: 'padloc_only' rows have no df_subtype, so they correctly drop out here.
_df_map_rows = name_map[name_map["source"] != "exclude"].dropna(subset=["df_subtype"])
df_subtype_to_canonical = _df_map_rows.set_index("df_subtype")["canonical_name"].to_dict()

print(f"DF subtype → canonical lookup: {len(df_subtype_to_canonical)} entries")
# Spot-check the key systems from the published paper
for key in ["SspBCDE", "Gao_Qat", "RM_Type_I", "BREX_I", "NARP1"]:
    print(f"  {key!r:20s} → {df_subtype_to_canonical.get(key, 'MISSING')}")

### Parse defence systems (activity == "Defense")

The loop structure is: **species → genome subdirectory → read TSV → tag with genome_id and species**.

Missing or empty TSVs (genomes with zero defence systems) produce zero rows — this is correct, not an error. After the loop we concatenate all species into one long-form table and apply the canonical name mapping.

Unmapped subtypes (not in `system_name_map`) are dropped with a warning. This should not happen if the map was built correctly from the full dataset, but we surface it explicitly rather than silently discarding data.

In [ ]:
df_defense_records = []   # accumulates one mini-dataframe per genome
df_missing = []           # tracks missing TSV files (should be zero)

for sp in SPECIES:
    df_dir = INTERIM / sp / "defensefinder"  # e.g. data/interim/abaumannii/defensefinder/

    for genome_dir in sorted(df_dir.iterdir()):  # each subdirectory = one genome
        if not genome_dir.is_dir():
            continue  # skip .DS_Store and other non-directory files

        tsv_path = genome_dir / "defense_finder_systems.tsv"

        if not tsv_path.exists():
            df_missing.append((sp, genome_dir.name))  # log, don't crash
            continue

        raw = pd.read_csv(tsv_path, sep="\t")

        if raw.empty:
            continue  # genome has zero defence systems — valid result

        # Keep defence rows only; anti-defence is parsed in the next cell
        defense_rows = raw[raw["activity"] == "Defense"][["subtype"]].copy()

        if defense_rows.empty:
            continue

        # Tag with canonical genome_id and species
        defense_rows["genome_id"] = norm_acc(genome_dir.name)  # GCF_000505685_1 → GCF_000505685.1
        defense_rows["species"]   = sp

        df_defense_records.append(defense_rows)

# ── Concatenate all species ─────────────────────────────────────────────────
if df_defense_records:
    df_defense_long = pd.concat(df_defense_records, ignore_index=True)
else:
    df_defense_long = pd.DataFrame(columns=["subtype", "genome_id", "species"])

print(f"DefenseFinder defence rows (raw, before name mapping): {len(df_defense_long):,}")
print(f"Missing TSV files: {len(df_missing)}")
if df_missing:
    print("  ", df_missing[:10])

In [ ]:
# ── Apply canonical name mapping ─────────────────────────────────────────────
# Map df_subtype → canonical_name using the lookup built above.
# Rows whose subtype is not in the map get NaN in canonical_name;
# we surface those as warnings, then drop them.
df_defense_long["canonical_name"] = df_defense_long["subtype"].map(df_subtype_to_canonical)

unmapped = df_defense_long[df_defense_long["canonical_name"].isna()]
if not unmapped.empty:
    print(f"WARNING: {len(unmapped)} rows have unmapped subtypes:")
    print(unmapped["subtype"].value_counts().head(20).to_string())
else:
    print("All DF defence subtypes mapped successfully.")

# Drop unmapped rows
df_defense_long = df_defense_long.dropna(subset=["canonical_name"])

print(f"\nRows after mapping: {len(df_defense_long):,}")
print(f"Unique canonical systems detected: {df_defense_long['canonical_name'].nunique()}")
print(f"Unique genomes with >=1 defence system: {df_defense_long['genome_id'].nunique()}")
print()
print("Top 10 most frequent defence systems:")
print(df_defense_long["canonical_name"].value_counts().head(10).to_string())

### Parse anti-defence systems (activity == "Antidefense")

Same loop, different filter. Anti-defence systems (e.g. `NARP1`, `Ocr`) are phage-encoded proteins that neutralise host defences. They appear in bacterial genomes when carried on prophages or MGEs.

These map to entries in `system_name_map` where `source == 'df_antidefense'`. The lookup dictionary built above already includes these rows.

In [ ]:
df_antidefense_records = []

for sp in SPECIES:
    df_dir = INTERIM / sp / "defensefinder"

    for genome_dir in sorted(df_dir.iterdir()):
        if not genome_dir.is_dir():
            continue  # skip .DS_Store

        tsv_path = genome_dir / "defense_finder_systems.tsv"

        if not tsv_path.exists():
            continue

        raw = pd.read_csv(tsv_path, sep="\t")

        if raw.empty:
            continue

        antidefense_rows = raw[raw["activity"] == "Antidefense"][["subtype"]].copy()

        if antidefense_rows.empty:
            continue

        antidefense_rows["genome_id"] = norm_acc(genome_dir.name)
        antidefense_rows["species"]   = sp

        df_antidefense_records.append(antidefense_rows)

if df_antidefense_records:
    df_antidefense_long = pd.concat(df_antidefense_records, ignore_index=True)
else:
    df_antidefense_long = pd.DataFrame(columns=["subtype", "genome_id", "species"])

# Map to canonical names
df_antidefense_long["canonical_name"] = df_antidefense_long["subtype"].map(df_subtype_to_canonical)

unmapped_adf = df_antidefense_long[df_antidefense_long["canonical_name"].isna()]
if not unmapped_adf.empty:
    print(f"WARNING: {len(unmapped_adf)} unmapped anti-defence subtypes:")
    print(unmapped_adf["subtype"].value_counts().head(10).to_string())

df_antidefense_long = df_antidefense_long.dropna(subset=["canonical_name"])

print(f"Anti-defence rows: {len(df_antidefense_long):,}")
print(f"Unique anti-defence systems: {df_antidefense_long['canonical_name'].nunique()}")
print(f"Genomes carrying >=1 anti-defence system: {df_antidefense_long['genome_id'].nunique()}")
print()
print("Top anti-defence systems:")
print(df_antidefense_long["canonical_name"].value_counts().head(10).to_string())

### Section 1 outputs\n\n`df_defense_long` and `df_antidefense_long` are long-form tables — one row per system **instance** per genome. They are not pivoted to wide here.\n\nWhy defer the pivot? Because the wide table needs to merge DefenseFinder and PADLOC calls for the same canonical system (e.g. both tools may detect `AbiC` in the same genome). That merge requires both long-form tables to be in memory simultaneously. Section 3 does this.\n\nThe anti-defence long-form will be pivoted in Section 3 directly (no PADLOC counterpart — PADLOC does not predict anti-defence systems).

In [ ]:
# Section 1 summary — print shapes of long-form tables
print(f"df_defense_long    : {len(df_defense_long):,} rows | "
      f"{df_defense_long['genome_id'].nunique()} genomes | "
      f"{df_defense_long['canonical_name'].nunique()} unique systems")

print(f"df_antidefense_long: {len(df_antidefense_long):,} rows | "
      f"{df_antidefense_long['genome_id'].nunique()} genomes | "
      f"{df_antidefense_long['canonical_name'].nunique()} unique anti-defence systems")

print()
print("Top 10 defence systems (by instance count across all genomes):")
print(df_defense_long["canonical_name"].value_counts().head(10).to_string())

print()
print("Top 5 anti-defence systems:")
print(df_antidefense_long["canonical_name"].value_counts().head(5).to_string())

### Verify: coverage across all 900 genomes

Every genome that has a DefenseFinder output directory should appear in at least one of the two tables (defense or anti-defence). Genomes with zero systems of either type simply won't appear in either table — they get zero-filled when we join everything later.

This cell checks how many distinct genome IDs appeared in the DF output, and whether that matches the expected 900.

In [ ]:
# All genome_ids seen across defence + antidefence
all_df_genomes = set(df_defense_long["genome_id"]) | set(df_antidefense_long["genome_id"])
print(f"Genomes with >=1 DF hit (defence or anti-defence): {len(all_df_genomes)}")

# Build the full 900-genome ID list from the directory structure
all_genome_ids = []
for sp in SPECIES:
    df_dir = INTERIM / sp / "defensefinder"
    for genome_dir in sorted(df_dir.iterdir()):
        if not genome_dir.is_dir():
            continue  # skip .DS_Store
        all_genome_ids.append(norm_acc(genome_dir.name))

all_genome_ids = sorted(set(all_genome_ids))
print(f"Total genome IDs from directory scan:               {len(all_genome_ids)}")

zero_hit_genomes = set(all_genome_ids) - all_df_genomes
print(f"Genomes with zero DF hits (pure zeros):             {len(zero_hit_genomes)}")

## Section 2 — Parse PADLOC outputs

### Why this step and what is different from DefenseFinder

PADLOC and DefenseFinder use the same underlying biology but different HMM databases and system boundary definitions. Running both tools and merging the results is standard practice in the field — systems detected by both tools are higher-confidence calls than those detected by only one.

The critical structural difference from DF: **PADLOC outputs one row per gene (protein HMM hit), not one row per system instance.** A single AbiL system contains two proteins (AbiLi and AbiLii), so it produces two rows, both with `system.number = 1`. If a genome has two AbiL copies, all four gene rows appear — two with `system.number = 1` and two with `system.number = 2`.

To get the instance count correctly, we deduplicate on `(system, system.number)` before building the long-form. This collapses multi-gene rows for the same system instance into one row.

### Mapping PADLOC system names → canonical names

Same principle as Section 1: `system_name_map.csv` contains a `padloc_system` column that maps PADLOC names to `canonical_name`. For `padloc_only` systems, the canonical name carries no tool prefix (e.g. `AVAST_II`, not `padloc_AVAST_II`). This is by design — the canonical name is the biology, not the tool.

After mapping, the canonical names in `padloc_long` and `df_defense_long` use the **same** vocabulary, so Section 3's merge is a straightforward union on `canonical_name`.

In [ ]:
# ── Build padloc_system → canonical_name lookup ──────────────────────────────
# Exclude 'exclude' rows and rows with no padloc_system entry (df_only, df_antidefense).
_padloc_map_rows = name_map[name_map["source"] != "exclude"].dropna(subset=["padloc_system"])
padloc_system_to_canonical = _padloc_map_rows.set_index("padloc_system")["canonical_name"].to_dict()

print(f"PADLOC system → canonical lookup: {len(padloc_system_to_canonical)} entries")
for key in ["AbiC", "PT_SspABCD", "qatABCD", "brex_type_I", "AVAST_type_II"]:
    print(f"  {key!r:22s} → {padloc_system_to_canonical.get(key, 'MISSING')}")

In [ ]:
padloc_records = []
padloc_missing  = []   # files absent entirely
padloc_unmapped_subtypes = []  # system names not in name_map

for sp in SPECIES:
    padloc_dir = INTERIM / sp / "padloc"  # flat directory — one CSV per genome

    for csv_file in sorted(padloc_dir.iterdir()):
        # PADLOC files: GCF_000505685_1_padloc.csv  (and .gff, .faa, .domtblout)
        if not csv_file.name.endswith("_padloc.csv"):
            continue

        # Reconstruct genome_id: strip the trailing '_padloc' before normalising
        stem = csv_file.stem.removesuffix("_padloc")   # GCF_000505685_1_padloc → GCF_000505685_1
        genome_id = norm_acc(stem)                       # GCF_000505685_1 → GCF_000505685.1

        raw = pd.read_csv(csv_file)

        if raw.empty:
            continue  # genome has zero PADLOC hits — valid

        # Deduplicate to one row per system INSTANCE:
        # system.number identifies distinct occurrences of the same system type within a genome.
        # Multiple gene rows with the same system.number belong to the same instance.
        instances = raw.drop_duplicates(subset=["system", "system.number"])[["system"]].copy()

        if instances.empty:
            continue

        instances["genome_id"] = genome_id
        instances["species"]   = sp

        padloc_records.append(instances)

# ── Concatenate ──────────────────────────────────────────────────────────────
if padloc_records:
    padloc_long_raw = pd.concat(padloc_records, ignore_index=True)
else:
    padloc_long_raw = pd.DataFrame(columns=["system", "genome_id", "species"])

print(f"PADLOC rows after instance deduplication (raw): {len(padloc_long_raw):,}")
print(f"Unique PADLOC system names seen: {padloc_long_raw['system'].nunique()}")
print(f"Genomes with >=1 PADLOC hit: {padloc_long_raw['genome_id'].nunique()}")

In [ ]:
# ── Apply canonical name mapping ─────────────────────────────────────────────
padloc_long_raw["canonical_name"] = padloc_long_raw["system"].map(padloc_system_to_canonical)

unmapped_padloc = padloc_long_raw[padloc_long_raw["canonical_name"].isna()]
if not unmapped_padloc.empty:
    print(f"WARNING: {len(unmapped_padloc)} rows with unmapped PADLOC system names:")
    print(unmapped_padloc["system"].value_counts().head(20).to_string())
    print()
else:
    print("All PADLOC system names mapped successfully.")

padloc_long = padloc_long_raw.dropna(subset=["canonical_name"]).copy()

print(f"Rows after mapping: {len(padloc_long):,}")
print(f"Unique canonical systems from PADLOC: {padloc_long['canonical_name'].nunique()}")
print(f"Genomes with >=1 mapped PADLOC system: {padloc_long['genome_id'].nunique()}")
print()
print("Top 10 PADLOC systems (by instance count):")
print(padloc_long["canonical_name"].value_counts().head(10).to_string())
print()

# ── Cross-tool sanity check ───────────────────────────────────────────────────
# Systems in BOTH tools' long-form tables: these are the 'both' rows in name_map.
# They should show up in both df_defense_long and padloc_long.
df_systems   = set(df_defense_long["canonical_name"].unique())
padloc_sys   = set(padloc_long["canonical_name"].unique())
in_both      = df_systems & padloc_sys
df_only_seen = df_systems - padloc_sys
padloc_only_seen = padloc_sys - df_systems

print(f"Systems seen by BOTH tools:    {len(in_both)}")
print(f"Systems seen by DF only:       {len(df_only_seen)}")
print(f"Systems seen by PADLOC only:   {len(padloc_only_seen)}")

## Section 3 — Merge DF + PADLOC → defence feature block

### Strategy

Both long-form tables share the same canonical name vocabulary (resolved in Sections 1–2). The merge proceeds in three stages:

**Stage A — count instances per (genome, system, tool).**  
Tag each long-form table with its tool name, concatenate, then groupby (genome_id, canonical_name, tool) and count rows. This gives DF and PADLOC instance counts side by side.

**Stage B — compute presence, count, n_tools per (genome, system).**  
- `presence` = 1 if either tool's count > 0 (union)  
- `count` = max(df_count, padloc_count) — takes the higher estimate without forcing a primary tool  
- `n_tools` = how many tools detected it (0, 1, or 2); used as a quality/confidence column, not a primary ML feature  

**Stage C — pivot to wide, reindex, sparsity filter.**  
Pivot `presence` to get a genome × system binary matrix. Reindex against the full 900-genome list (fills absent genomes with 0). Drop any system column present in <5 genomes across all 900 — these are too rare to be informative and add noise to distance metrics.

### Anti-defence: simpler

PADLOC does not predict anti-defence systems. `df_antidefense_long` goes straight to a pivot — no merge needed. Same sparsity filter applied.

In [ ]:
# ── Stage A: count instances per (genome, system, tool) ─────────────────────
# Tag each table with its source tool before concatenating
combined_long = pd.concat([
    df_defense_long.assign(tool="df"),
    padloc_long.assign(tool="padloc"),
], ignore_index=True)

# groupby gives us: for each (genome, system, tool), how many instances?
tool_counts = (
    combined_long
    .groupby(["genome_id", "canonical_name", "tool"])
    .size()                             # row count = instance count
    .reset_index(name="count")
)

# Pivot tool axis: index = (genome_id, canonical_name), columns = tool names
counts_by_tool = tool_counts.pivot_table(
    index=["genome_id", "canonical_name"],
    columns="tool",
    values="count",
    fill_value=0,       # 0 means that tool detected nothing for this (genome, system)
).reset_index()

# Rename columns for clarity (pivot creates a MultiIndex-like columns object)
counts_by_tool.columns.name = None
for col in ["df", "padloc"]:       # ensure both columns exist even if one tool had no hits
    if col not in counts_by_tool.columns:
        counts_by_tool[col] = 0

print(f"(genome, system) pairs detected by at least one tool: {len(counts_by_tool):,}")
print(f"Preview:")
print(counts_by_tool.head(6).to_string())

In [ ]:
# ── Stage B: compute presence, count, n_tools ────────────────────────────────
counts_by_tool["presence"] = 1   # every row here is a detected (genome, system) pair
counts_by_tool["count"]    = counts_by_tool[["df", "padloc"]].max(axis=1)
counts_by_tool["n_tools"]  = (
    (counts_by_tool["df"] > 0).astype(int) +
    (counts_by_tool["padloc"] > 0).astype(int)
)

print("n_tools distribution (how many tools agreed on each (genome, system) detection):")
print(counts_by_tool["n_tools"].value_counts().sort_index().to_string())
print()

# Spot-check: key systems from the published paper
for sys in ["SspBCDE", "Gao_Qat", "RM_Type_I", "AbiE"]:
    subset = counts_by_tool[counts_by_tool["canonical_name"] == sys]
    n2 = (subset["n_tools"] == 2).sum()
    n1 = (subset["n_tools"] == 1).sum()
    total = len(subset)
    print(f"  {sys:20s}: {total} genomes | both tools={n2} | one tool={n1}")

In [ ]:
# ── Stage C: pivot to wide, reindex, sparsity filter ─────────────────────────

# Pivot presence → genome × system binary matrix
defence_pa = (
    counts_by_tool
    .pivot(index="genome_id", columns="canonical_name", values="presence")
    .fillna(0)          # genomes absent from this system's rows → 0
    .astype(int)
)

# Pivot count → genome × system count matrix
defence_count = (
    counts_by_tool
    .pivot(index="genome_id", columns="canonical_name", values="count")
    .fillna(0)
    .astype(int)
)

# Reindex both against the full 900-genome list
# Genomes that had ZERO systems in both tools get all-zero rows
defence_pa    = defence_pa.reindex(all_genome_ids, fill_value=0)
defence_count = defence_count.reindex(all_genome_ids, fill_value=0)

print(f"Before sparsity filter: {defence_pa.shape[1]} defence systems")

# Sparsity filter: drop systems present in < 5 genomes
system_prevalence = defence_pa.sum(axis=0)   # number of genomes with presence=1
keep_systems = system_prevalence[system_prevalence >= 5].index

defence_pa    = defence_pa[keep_systems]
defence_count = defence_count[keep_systems]

print(f"After sparsity filter (>=5 genomes): {defence_pa.shape[1]} defence systems")
print(f"Systems dropped (too rare): {len(system_prevalence) - len(keep_systems)}")
print(f"Defence P/A matrix shape: {defence_pa.shape}")
print()
print("Sparsity of final defence P/A matrix:")
print(f"  {(defence_pa == 0).values.mean():.1%}")

In [ ]:
# ── Anti-defence: straight pivot (DF only, no merge needed) ─────────────────
adef_pa = (
    df_antidefense_long
    .groupby(["genome_id", "canonical_name"])
    .size()
    .unstack(fill_value=0)
    .clip(upper=1)                       # presence/absence
    .reindex(all_genome_ids, fill_value=0)
)

# Sparsity filter — same threshold
adef_prevalence = adef_pa.sum(axis=0)
keep_adef = adef_prevalence[adef_prevalence >= 5].index
adef_pa = adef_pa[keep_adef]

print(f"Anti-defence P/A matrix: {adef_pa.shape}  (after sparsity filter)")
print(f"Anti-defence systems dropped: {len(adef_prevalence) - len(keep_adef)}")
print()

# ── Summary of defence feature block ─────────────────────────────────────────
print("=" * 55)
print("DEFENCE FEATURE BLOCK SUMMARY")
print("=" * 55)
print(f"Genomes (rows)         : {defence_pa.shape[0]}")
print(f"Defence systems (cols) : {defence_pa.shape[1]}")
print(f"Anti-defence (cols)    : {adef_pa.shape[1]}")
print(f"Total feature columns  : {defence_pa.shape[1] + adef_pa.shape[1]}")
print()
# Verify key systems from published paper survived the filter
for sys in ["SspBCDE", "Gao_Qat", "RM_Type_I", "RM_Type_II", "RM_Type_IV"]:
    status = "PRESENT" if sys in defence_pa.columns else "DROPPED"
    if status == "PRESENT":
        prev = int(defence_pa[sys].sum())
        print(f"  {sys:20s}: {status} ({prev}/900 genomes)")
    else:
        print(f"  {sys:20s}: {status}")

## Section 4 — Parse ResFinder outputs (ARG counts)

### Why ARG count is the Q2 target, not just another feature

The published *Acinetobacter* paper found that RM systems negatively correlate with ARG count — RM acts as a restriction barrier against incoming MGEs that carry ARGs. Q2 asks whether this generalises: can the defence system profile predict high-ARG-burden genomes across ESKAPE?

To test this, we need ARG count as a *label* (the thing we're predicting) and as a *feature* (correlation context). Both go in the matrix; the Q2 target variable is built from the ARG count column in Section 9.

### What counts as one ARG

ResFinder outputs one row per HMM hit — if `blaTEM-1` appears on three plasmids, there are three rows. We compute two ARG metrics:

- `arg_count_unique`: distinct gene names per genome — "how many different resistance genes does this genome carry?" This is the primary burden metric and the Q2 label basis.
- `arg_count_total`: total hits per genome — includes multi-copy genes, a proxy for MGE load.

Both are reported. For Q2, we use `arg_count_unique`.

### File location

`data/interim/{species}/resfinder/{accession}/ResFinder_results_tab.txt` — tab-separated, header row present.

In [ ]:
arg_records = []   # one dict per genome

for sp in SPECIES:
    res_dir = INTERIM / sp / "resfinder"   # each genome has its own subdirectory

    for genome_dir in sorted(res_dir.iterdir()):
        if not genome_dir.is_dir():
            continue  # skip .DS_Store

        results_file = genome_dir / "ResFinder_results_tab.txt"

        if not results_file.exists():
            # ResFinder was called but produced no output — treat as zero ARGs
            arg_records.append({
                "genome_id": norm_acc(genome_dir.name),
                "species": sp,
                "arg_count_unique": 0,
                "arg_count_total": 0,
            })
            continue

        raw = pd.read_csv(results_file, sep="\t")

        # ResFinder header: Resistance gene | Identity | ... | Phenotype | ...
        # Column 0 is the gene name — strip it from 'Resistance gene' header
        # Some versions use slightly different headers; guard with fallback
        if "Resistance gene" in raw.columns:
            gene_col = "Resistance gene"
        else:
            gene_col = raw.columns[0]

        genome_id = norm_acc(genome_dir.name)

        if raw.empty:
            arg_records.append({
                "genome_id": genome_id, "species": sp,
                "arg_count_unique": 0, "arg_count_total": 0,
            })
            continue

        arg_records.append({
            "genome_id": genome_id,
            "species":   sp,
            "arg_count_unique": raw[gene_col].nunique(),  # distinct gene names
            "arg_count_total":  len(raw),                  # all hits including duplicates
        })

arg_df = pd.DataFrame(arg_records).set_index("genome_id")

print(f"ARG records parsed: {len(arg_df):,}  (expect 900)")
print(f"Genomes with >=1 ARG: {(arg_df['arg_count_unique'] > 0).sum()}")
print(f"Genomes with zero ARGs: {(arg_df['arg_count_unique'] == 0).sum()}")
print()
print("arg_count_unique distribution:")
print(arg_df["arg_count_unique"].describe().round(1).to_string())
print()
print("Per-species median ARG count (unique):")
print(arg_df.groupby("species")["arg_count_unique"].median().sort_values(ascending=False).to_string())

## Section 5 — Parse ICEberg outputs (IME/ICE counts)

### Database structure and the coverage ambiguity

The ICEberg tBLASTn database contains protein sequences from known Integrative and Mobilizable Elements (IMEs). Multiple proteins from the same element share the same FASTA header prefix (and therefore the same BLAST `qseqid`). For example, all 5 proteins from element `ICEberg|10_IME` all produce hits with `qseqid = ICEberg|10_IME`.

This means `qstart`/`qend` from a BLAST hit are amino acid coordinates within *one* of those proteins, but the output does not record *which*. To apply the pre-registered 80% query coverage filter, we:
1. Parse the ICEberg FASTA to find the **maximum protein length** per element ID.
2. Use that maximum as the coverage denominator: `coverage = (qend - qstart + 1) / max_protein_length`.

This *underestimates* coverage — a short-protein hit will appear to have lower coverage than it actually does against its actual protein. This is the conservative direction: some genuine hits may be filtered, but false positives are suppressed.

### Counting logic

Consistent with the design decision: `ime_count_unique` = number of distinct `qseqid` values (element IDs) that passed the filter per genome. This counts how many different ICE/IME elements are represented. `ime_count_total` = total filtered hits.

In [ ]:
from collections import defaultdict

def parse_fasta_max_lengths(fasta_path: Path) -> dict:
    """
    Parse a protein FASTA and return {first_word_of_header: max_sequence_length_aa}.
    When multiple sequences share the same first header word (as in ICEberg),
    the maximum length is kept — used as the conservative coverage denominator.
    """
    lengths = defaultdict(list)
    current_id = None
    current_len = 0
    with open(fasta_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current_id is not None:
                    lengths[current_id].append(current_len)
                current_id = line[1:].split()[0]   # first word after '>'
                current_len = 0
            else:
                current_len += len(line)           # amino acid characters
    if current_id is not None:
        lengths[current_id].append(current_len)
    return {k: max(v) for k, v in lengths.items()}


# ── Build ICEberg protein length lookup ──────────────────────────────────────
ICEBERG_FASTA = ROOT / "data" / "raw" / "databases" / "ICEberg_IME.fasta"
ice_lengths = parse_fasta_max_lengths(ICEBERG_FASTA)

print(f"ICEberg unique element IDs with protein lengths: {len(ice_lengths)}")
# Spot-check: element 160 appears in the sample BLAST output
for eid in ["ICEberg|160_IME", "ICEberg|10_IME", "ICEberg|1_ICE"]:
    print(f"  {eid}: max protein length = {ice_lengths.get(eid, 'NOT FOUND')} aa")

In [ ]:
BLAST_COLS = ["qseqid", "sseqid", "pident", "length", "mismatch",
              "gapopen", "qstart", "qend", "sstart", "send", "evalue", "bitscore"]

PIDENT_MIN    = 40.0   # % identity threshold
COVERAGE_MIN  = 0.80   # 80% query coverage

ime_records = []

for sp in SPECIES:
    ice_dir = INTERIM / sp / "iceberg"

    for tsv_file in sorted(ice_dir.iterdir()):
        if not tsv_file.name.endswith("_iceberg.tsv"):
            continue

        stem      = tsv_file.stem.removesuffix("_iceberg")
        genome_id = norm_acc(stem)

        # Empty file = zero ICEberg hits (legitimate result)
        if tsv_file.stat().st_size == 0:
            ime_records.append({
                "genome_id": genome_id, "species": sp,
                "ime_count_unique": 0, "ime_count_total": 0,
            })
            continue

        raw = pd.read_csv(tsv_file, sep="\t", header=None, names=BLAST_COLS)

        # ── Apply filters ────────────────────────────────────────────────────
        # 1. pident >= 40%
        raw = raw[raw["pident"] >= PIDENT_MIN]

        # 2. Query coverage >= 80%, using max protein length as denominator
        #    coverage = (qend - qstart + 1) / max_protein_length
        #    Hits with unknown qseqid (not in our length dict) are dropped.
        raw = raw[raw["qseqid"].isin(ice_lengths)]
        raw = raw.copy()
        raw["max_prot_len"] = raw["qseqid"].map(ice_lengths)
        raw["coverage"]     = (raw["qend"] - raw["qstart"] + 1) / raw["max_prot_len"]
        raw = raw[raw["coverage"] >= COVERAGE_MIN]

        ime_records.append({
            "genome_id":       genome_id,
            "species":         sp,
            "ime_count_unique": raw["qseqid"].nunique(),  # distinct element IDs
            "ime_count_total":  len(raw),                  # all passing hits
        })

ime_df = pd.DataFrame(ime_records).set_index("genome_id")

print(f"IME records: {len(ime_df):,}  (expect 900)")
print(f"Genomes with >=1 IME hit: {(ime_df['ime_count_unique'] > 0).sum()}")
print(f"Genomes with zero IMEs:   {(ime_df['ime_count_unique'] == 0).sum()}")
print()
print("ime_count_unique distribution:")
print(ime_df["ime_count_unique"].describe().round(1).to_string())
print()
print("Per-species median IME count (unique):")
print(ime_df.groupby("species")["ime_count_unique"].median().sort_values(ascending=False).to_string())

## Section 6 — Parse BacMet outputs (HMRG counts)

BacMet (Heavy Metal Resistance Gene database) records resistance genes for metals: arsenic, copper, mercury, zinc, cobalt, etc. Heavy metal resistance genes are co-selected with ARGs on the same MGEs — metal contamination in the environment selects for resistance cassettes that carry both.

The BLAST structure is identical to ICEberg, but BacMet has **unique qseqids per protein** (`BAC0001|abeM|...`), so query coverage is exact: `coverage = (qend - qstart + 1) / protein_length`.

Same BLAST filters: pident ≥ 40%, coverage ≥ 80%.

Counting: `hmrg_count_unique` = distinct BacMet protein IDs passing filters. `hmrg_count_total` = all passing hits.

In [ ]:
BACMET_FASTA = ROOT / "data" / "raw" / "databases" / "BacMet2_EXP_database.fasta"

# BacMet: each protein has a unique qseqid → exact coverage calculation
# parse_fasta_max_lengths works here too: max of a single value = the value itself
bacmet_lengths = parse_fasta_max_lengths(BACMET_FASTA)

print(f"BacMet unique protein IDs: {len(bacmet_lengths)}")
for pid in ["BAC0001|abeM|tr|Q5FAM9|Q5FAM9_ACIBA",
            "BAC0002|abeS|tr|Q2FD83|Q2FD83_ACIBA",
            "BAC0005|acrA|sp|P0AE06|ACRA_ECOLI"]:
    print(f"  {pid[:45]}: {bacmet_lengths.get(pid, 'NOT FOUND')} aa")

print()

hmrg_records = []

for sp in SPECIES:
    bm_dir = INTERIM / sp / "bacmet"

    for tsv_file in sorted(bm_dir.iterdir()):
        if not tsv_file.name.endswith("_bacmet.tsv"):
            continue

        stem      = tsv_file.stem.removesuffix("_bacmet")
        genome_id = norm_acc(stem)

        if tsv_file.stat().st_size == 0:
            hmrg_records.append({
                "genome_id": genome_id, "species": sp,
                "hmrg_count_unique": 0, "hmrg_count_total": 0,
            })
            continue

        raw = pd.read_csv(tsv_file, sep="\t", header=None, names=BLAST_COLS)

        # pident filter
        raw = raw[raw["pident"] >= PIDENT_MIN]

        # Coverage filter — exact for BacMet (one protein per qseqid)
        raw = raw[raw["qseqid"].isin(bacmet_lengths)]
        raw = raw.copy()
        raw["prot_len"]  = raw["qseqid"].map(bacmet_lengths)
        raw["coverage"]  = (raw["qend"] - raw["qstart"] + 1) / raw["prot_len"]
        raw = raw[raw["coverage"] >= COVERAGE_MIN]

        hmrg_records.append({
            "genome_id":        genome_id,
            "species":          sp,
            "hmrg_count_unique": raw["qseqid"].nunique(),
            "hmrg_count_total":  len(raw),
        })

hmrg_df = pd.DataFrame(hmrg_records).set_index("genome_id")

print(f"HMRG records: {len(hmrg_df):,}  (expect 900)")
print(f"Genomes with >=1 HMRG: {(hmrg_df['hmrg_count_unique'] > 0).sum()}")
print(f"Genomes with zero HMRGs: {(hmrg_df['hmrg_count_unique'] == 0).sum()}")
print()
print("hmrg_count_unique distribution:")
print(hmrg_df["hmrg_count_unique"].describe().round(1).to_string())
print()
print("Per-species median HMRG count (unique):")
print(hmrg_df.groupby("species")["hmrg_count_unique"].median().sort_values(ascending=False).to_string())

### BacMet caveat — high counts expected, interpretation is conservative

All 900 genomes have ≥1 HMRG hit; median is 221 distinct BacMet proteins per genome. This is not a pipeline error.

The BacMet EXP database includes many **chromosomally encoded housekeeping proteins** with metal-handling functions (RND efflux pumps, copper-transporting ATPases, arsenic detoxification enzymes) that are broadly conserved across bacteria. At pident ≥ 40%, distant homologs match in virtually every genome.

Implications:
- `hmrg_count_unique` does not cleanly separate "has acquired heavy metal resistance" from "has normal bacterial metal homeostasis machinery." It reflects both.
- This makes HMRG count a weaker proxy for MGE-acquired metal resistance than ARG count is for antibiotic resistance, because ResFinder's curated acquired-gene focus is stricter than BacMet's experimentally verified proteins.
- For ML features: `hmrg_count_unique` will likely carry species-level information (gram-positive vs gram-negative baseline metal homeostasis genes differ) more than MGE-load information. Flag in the Methods/Discussion.
- The correlation with ARG/IME counts (the paper's main question) is still testable — it just should be interpreted as "co-occurrence of metal + antibiotic resistance gene content" not as "co-acquisition on the same MGE."